# PhysIQ – Task 3: Causal Chain Reasoning

Given a 2D physics scenario with multiple interacting objects, describe the chain of causal events that will occur and the final outcome.

**Format:**
```
STEPS:
1. <object_A> hits <object_B>
2. <object_B> knocks <object_C>
...
OUTCOME: <final state description>
```
**Scoring:** Final outcome match (0.5) + intermediate steps (0.5)

In [ ]:
# ── Cell 1: Setup & imports ──────────────────────────────────────────────────
import sys, os, json, re
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'Kaggle_agi_bench'))

import kaggle_benchmarks as kbench
print('kaggle-benchmarks', kbench.__version__)

In [ ]:
# ── Cell 2: Physics engine & scenario library ────────────────────────────────
from physiq.engine import PhysIQWorld
from physiq.formats import build_prompt
from physiq.generation import compute_ground_truth, validate_scenario, _task_seed
from physiq.scoring import score_causal_chain, outcome_matches
from physiq.templates.causal_chain import CAUSAL_CHAIN_TEMPLATES
from physiq.templates import SCENARIO_COUNTS

print(f'Causal chain templates: {len(CAUSAL_CHAIN_TEMPLATES)}')

In [ ]:
# ── Cell 3: Generate causal chain scenarios ──────────────────────────────────
MASTER_SEED = 42
task_type = 'causal_chain'
counts = SCENARIO_COUNTS[task_type]

task_seed = _task_seed(task_type, MASTER_SEED)
rng = np.random.RandomState(task_seed)

scenarios = []
for difficulty in ['easy', 'medium', 'hard']:
    target = counts[difficulty]
    collected = []
    attempt = 0
    while len(collected) < target and attempt < target * 3:
        seed = int(rng.randint(0, 2**31))
        template = CAUSAL_CHAIN_TEMPLATES[attempt % len(CAUSAL_CHAIN_TEMPLATES)]
        attempt += 1
        try:
            s = template.generate(difficulty, seed)
        except Exception:
            continue
        if validate_scenario(s, task_type):
            s['_ground_truth'] = compute_ground_truth(s, task_type)
            collected.append(s)
    scenarios.extend(collected)
    print(f'  {difficulty}: {len(collected)}/{target}')

print(f'Total causal chain scenarios: {len(scenarios)}')

In [ ]:
# ── Cell 4: Build evaluation DataFrame ──────────────────────────────────────
rows = []
for s in scenarios:
    gt = s['_ground_truth']
    for fmt in ['json', 'ascii', 'nl']:
        rows.append({
            'scenario_id':    s['id'],
            'difficulty':     s['difficulty'],
            'representation': fmt,
            'prompt':         build_prompt(s, fmt, task_type),
            'ground_truth':   json.dumps(gt),
        })

df_causal = pd.DataFrame(rows)
print(df_causal.shape)
df_causal.head(2)

In [ ]:
# ── Cell 5: Scoring helpers ──────────────────────────────────────────────────
_INTERACTION_KWS = {
    'collision', 'collide', 'hit', 'hits', 'strike', 'impact',
    'launch', 'fall', 'falls', 'drop', 'push', 'slide', 'roll',
    'tip', 'topple', 'bounce', 'knock', 'trigger', 'catapult',
}


def _parse_causal_response(response: str) -> dict:
    """Parse STEPS: ... OUTCOME: ... format from model response."""
    lines = response.strip().split('\n')
    steps = []
    outcome = ''
    in_steps = False

    for line in lines:
        line_s = line.strip()
        if re.match(r'^STEPS\s*:', line_s, re.IGNORECASE):
            in_steps = True
            continue
        if re.match(r'^OUTCOME\s*:', line_s, re.IGNORECASE):
            in_steps = False
            outcome = re.sub(r'^OUTCOME\s*:\s*', '', line_s, flags=re.IGNORECASE)
            continue
        if in_steps and re.match(r'^\d+\.', line_s):
            steps.append(re.sub(r'^\d+\.\s*', '', line_s))

    # Fallback: extract any numbered list
    if not steps:
        for line in lines:
            m = re.match(r'^[\d]+[.):]\s*(.+)', line.strip())
            if m:
                steps.append(m.group(1))

    # Fallback outcome: last non-empty line
    if not outcome:
        for line in reversed(lines):
            if line.strip():
                outcome = line.strip()
                break

    return {'steps': steps, 'outcome': outcome}


def _event_step_match(step_text: str, event: dict) -> bool:
    """Check if a predicted step matches a simulated event."""
    text_lower = step_text.lower()
    # Any interaction keyword present?
    has_interaction = any(kw in text_lower for kw in _INTERACTION_KWS)
    # Any involved object mentioned?
    obj_ids = event.get('objects', [])
    has_object = any(oid.lower() in text_lower for oid in obj_ids)
    return has_interaction and has_object


def _outcome_matches(pred: str, actual: str) -> bool:
    """Fuzzy outcome match using token overlap."""
    pred_tokens = set(re.sub(r'[^a-z0-9]', ' ', pred.lower()).split())
    actual_tokens = set(re.sub(r'[^a-z0-9]', ' ', actual.lower()).split())
    if not actual_tokens:
        return False
    overlap = len(pred_tokens & actual_tokens) / len(actual_tokens)
    return overlap >= 0.4


def _score_causal_response(response: str, ground_truth_json: str) -> float:
    gt = json.loads(ground_truth_json)
    parsed = _parse_causal_response(response)

    score = 0.0

    # 1) Final outcome match (0.5)
    gt_outcome = gt.get('outcome', '')
    if parsed['outcome'] and _outcome_matches(parsed['outcome'], gt_outcome):
        score += 0.5

    # 2) Intermediate steps (0.5) — each matched event contributes proportionally
    gt_events = gt.get('events', [])
    if gt_events and parsed['steps']:
        n_events = len(gt_events)
        matched = 0
        used_events = set()
        for step in parsed['steps']:
            for i, ev in enumerate(gt_events):
                if i not in used_events and _event_step_match(step, ev):
                    matched += 1
                    used_events.add(i)
                    break
        score += 0.5 * (matched / n_events)

    return min(score, 1.0)


# Sanity check
gt_sample = df_causal.iloc[0]['ground_truth']
gt_dict = json.loads(gt_sample)
events = gt_dict.get('events', [])
steps_str = '\n'.join(f'{i+1}. {" ".join(e.get("objects", []))} collide' for i, e in enumerate(events[:3]))
fake = f'STEPS:\n{steps_str}\nOUTCOME: {gt_dict.get("outcome", "objects at rest")}'
print('Perfect answer score:', _score_causal_response(fake, gt_sample))

In [ ]:
# ── Cell 6: Task definition ──────────────────────────────────────────────────
@kbench.task(
    name='PhysIQ Causal Chain Reasoning',
    description=(
        'Given a multi-body 2D physics scenario, describe the chain of causal '
        'events and the final outcome. '
        'Tests multi-body cause-and-effect reasoning in physics.'
    ),
)
def physiq_causal_chain(llm: kbench.LLMChat, prompt: str, ground_truth: str) -> float:
    """Predict causal event chain in multi-body 2D physics scenario."""
    response = llm.prompt(prompt)
    return _score_causal_response(response, ground_truth)

In [ ]:
# ── Cell 7: Dry-run sanity check ─────────────────────────────────────────────
required_cols = {'prompt', 'ground_truth'}
assert required_cols.issubset(df_causal.columns)
print('DataFrame columns OK:', list(df_causal.columns))
print('Rows:', len(df_causal))
avg_events = np.mean([
    len(json.loads(gt).get('events', []))
    for gt in df_causal['ground_truth'].unique()
])
print(f'Avg events per scenario: {avg_events:.1f}')

In [ ]:
# ── Cell 8: Evaluation run ───────────────────────────────────────────────────
try:
    llm = kbench.llm
except AttributeError:
    print('No Kaggle LLM available (local run). Skipping evaluation.')
    llm = None

if llm is not None:
    results = []
    for _, row in df_causal.iterrows():
        run = physiq_causal_chain.run(
            llm=llm,
            prompt=row['prompt'],
            ground_truth=row['ground_truth'],
        )
        results.append({
            'scenario_id':    row['scenario_id'],
            'difficulty':     row['difficulty'],
            'representation': row['representation'],
            'score':          run.result,
        })
    df_results = pd.DataFrame(results)
    os.makedirs('../outputs', exist_ok=True)
    df_results.to_csv('../outputs/task3_causal_chain_results.csv', index=False)
    print('Mean score:', df_results['score'].mean())
    print(df_results.groupby(['difficulty', 'representation'])['score'].mean().unstack())

In [ ]:
# ── Cell 9: Results analysis ─────────────────────────────────────────────────
if llm is not None and 'df_results' in dir():
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df_results.groupby('difficulty')['score'].mean().plot(
        kind='bar', ax=axes[0], color=['#4CAF50', '#FF9800', '#F44336'])
    axes[0].set_title('Causal Chain Score by Difficulty')
    axes[0].set_ylabel('Mean Score (0-1)')
    axes[0].set_ylim(0, 1)

    df_results.groupby('representation')['score'].mean().plot(
        kind='bar', ax=axes[1], color=['#2196F3', '#9C27B0', '#00BCD4'])
    axes[1].set_title('Causal Chain Score by Format')
    axes[1].set_ylabel('Mean Score (0-1)')
    axes[1].set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig('../outputs/task3_causal_chain_results.png', dpi=150)
    plt.show()

In [ ]:
# ── Cell 10: Choose this task for the leaderboard ────────────────────────────
%choose physiq_causal_chain